In [0]:
catalog_name = "ct_oil_gas" 
silver_schema_name = "sc_silver"
silver_table_name = "oil_gas_transactions"
gold_schema_name = "sc_gold"
gold_table_name = "daily_demand"


In [0]:
df=  spark.table("{catalog_name}.{silver_schema_name}.{silver_table_name}")

In [0]:
from pyspark.sql.functions import col, sum as _sum, avg, round as _round, when

def dbl(expr, name, digits=2):
    return _round(expr, digits).cast("double").alias(name)

gold_df = (
    silver_df.groupBy("product_name", "transaction_date", "destination_city")
    .agg(
        dbl(_sum("demand_quantity"), "total_demand_quantity"),
        dbl(_sum("total_cost_usd"), "total_cost_usd"),
        dbl(avg("supplier_reliability_score"), "avg_supplier_reliability_score"),
        dbl(_sum("ordered_quantity"), "total_ordered_quantity"),
        dbl(_sum("available_inventory"), "total_available_inventory"),
        dbl(avg("unit_price_usd"), "avg_unit_price"),
        dbl(_sum("product_cost_usd"), "total_product_cost_usd"),
        dbl(_sum("transportation_cost_usd"), "total_transportation_cost_usd"),
        dbl(_sum("expected_lead_time_days"), "total_expected_lead_time_days"),
        dbl(_sum("actual_lead_time_days"), "total_actual_lead_time_days"),
        dbl(avg("quality_score"), "avg_quality_score"),
    )
    .withColumn(
        "total_delay_days",
        col("total_actual_lead_time_days") - col("total_expected_lead_time_days"),
    )
    .withColumn("is_delayed", when(col("total_delay_days") >= 1, 1).otherwise(0))
    .orderBy("transaction_date")
)

gold_df.printSchema()   # everything numeric should be double

In [0]:
# %sql
# select count(*) from 
# ct_oil_gas.sc_gold.oil_gas_transactions_aggregated

In [0]:
gold_df.write.format("delta") \
    .mode("overwrite") \
    .saveAsTable(f"{catalog_name}.{gold_schema_name}.{gold_table_name}")